In [ ]:
import numpy as np
from collections import Counter

class Node:
  def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
    self.feature = feature
    self.threshold = threshold
    self.left = left
    self.right = right
    self.value = value
    
class DecisionTreeScratch:
  def __init__(self, min_samples_split=2, max_depth=float("inf")):
    self.min_samples_split = min_samples_split
    self.max_depth = max_depth
    self.root = None
    
  def fit(self, X, y):
    self.root = self._grow_tree(X, y)
    
  def _grow_tree(self, X , y, depth=0):
    n_samples, n_features = X.shape
    n_labels = len(np.unique(y))
    
    # 1. Stopping Criteria
    # If pure (1 label), or max depth reached, or too few samples -> Make Leaf
    
    if(depth >=self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
      leaf_value = self._most_comman_label(y)
      return Node(value=leaf_value)
    
    # 2. Find the Best Split
    feat_idx, threshold = self._best_split(X, y, n_features)
    
    # 3. Create Child Nodes
    # If we couldn't find a split (rare), return leaf
    if feat_idx is None:
      return Node(value=self._most_comman_label(y))
    
    left_idxs = X[ :, feat_idx] < threshold
    right_idxs = ~left_idxs # Inverse boolean mask
    
    left = self._grow_tree(X[left_idxs], y[left_idxs], depth + 1)
    right = self._grow_tree(X[right_idxs], y[right_idxs], depth + 1)
    
    return Node(feature=feat_idx, threshold=threshold, left=left, right=right)
  
  def _best_split(self, X, y, n_features):
    best_gain = -1
    split_idx, split_threshold = None, None
    
    for feature_idx in range(n_features):
      X_column = X[:, feature_idx]
      thresholds = np.unique(X_column)
      
      for thr in thresholds:
        gain = self._information_gain(y, X_column, thr)
        
        if gain > best_gain:
          best_gain = gain
          split_idx = feature_idx
          split_threshold = thr
          
    return split_idx, split_threshold
  
  def _information_gain(self, y, X_column, threshold):
    # Parent Entropy
    parent_entropy = self._entropy(y)
    
    # Generate Split
    left_idxs = X_column < threshold
    right_idxs = ~left_idxs
    
    if(len(y[left_idxs])==0 or len(y[right_idxs])==0):
      return 0
    
    # Weighted Average Child Entropy
    n = len(y)
    n_l, n_r = len(y[left_idxs]) , len(y[right_idxs])
    e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
    
    child_entropy = (n_l/n) * e_l + (n_r/n) * e_r
    
    return parent_entropy - child_entropy
  
  def _entropy(self, y):
    hist = np.bincount(y)
    ps = hist/len(y)
    return -np.sum([p * np.log2(p) for p in ps if p > 0])
  
  def _most_comman_label(self, y):
    if len(y) ==0: return 0
    counter = Counter(y)
    return counter.most_common(1)[0][0]
  
  def predict(self, X):
    return np.array([self._traverse_tree(x, self.root) for x in X])
  
  def _traverse_tree(self, x, node):
    if node.value is not None:
      return node.value
    
    if x[node.feature] < node.threshold:
      return self._traverse_tree(x, node.left)
    
    return self._traverse_tree(x, node.right)

In [23]:
# --- Usage Example ---
if __name__ == "__main__":
    from sklearn.datasets import load_wine
    from sklearn.model_selection import train_test_split

    data = load_wine()
    X, y = data.data, data.target

    # This dataset has 3 classes (3 types of wine)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    clf = DecisionTreeScratch(max_depth=10)
    clf.fit(X_train, y_train)

    predictions = clf.predict(X_test)
    accuracy = np.sum(predictions == y_test) / len(y_test)
    print(f"Accuracy: {accuracy*100:.2f}%")

Accuracy: 91.67%
